Based on our example dataset with two classes (A and B), we have calculated the following values:

1. **Entropy**: The calculated entropy of the dataset is approximately 0.971. This value indicates a moderate level of disorder in the dataset, considering that it's not very close to 0 (which would mean no disorder) and not at its maximum (which would mean complete disorder for a binary classification).

2. **Gini Impurity**: The Gini impurity for the dataset is 0.48. This value, being less than 0.5, suggests some level of purity in the dataset but still indicates a mix of classes A and B.

3. **Information Gain**: The information gain from the chosen split is 0.0. This result implies that the split did not reduce the entropy or disorder of the dataset. In other words, the split did not add any additional information that could help distinguish between classes A and B more effectively than before.

These metrics provide insight into the nature of the dataset and the effectiveness of potential splits when constructing a decision tree. In practical applications, you would use these calculations to choose the best feature and split at each node in the tree to maximize the purity of the subsets created.

To understand these three terms, you have to think like a **Decision Tree**. A Decision Tree’s only goal is to take a messy mix of data and split it into groups that are as "pure" as possible (e.g., all Apples in one bucket, all Oranges in another).

These metrics are the "rulers" we use to measure how messy those buckets are.

---

### 1. Entropy (The "Chaos" Meter)

Entropy comes from information theory. It measures the level of **disorder** or **uncertainty** in a group.

* **How it works:** * If a bucket has 50% Class A and 50% Class B, Entropy is **1.0** (Maximum chaos/Perfectly mixed).
* If a bucket has 100% Class A and 0% Class B, Entropy is **0.0** (Zero chaos/Perfectly pure).


* **Your Value (0.971):** This is very close to 1.0. It means your dataset is currently a "jumbled mess" where the classes are almost evenly mixed.

---

### 2. Gini Impurity (The "Mistake" Probability)

Gini is the default metric used by the **CART** algorithm (Scikit-Learn's default). It measures the probability that a randomly chosen element from the set would be incorrectly labeled if it was randomly labeled according to the distribution of labels in the subset.

* **How it works:**
* **0.0** means the node is "Pure" (all one class).
* **0.5** (for binary classes) is the "Dirtiest" it can be.


* **Your Value (0.48):** Since 0.5 is the worst possible score for two classes, your score of 0.48 tells us the dataset is almost perfectly impure. If you closed your eyes and guessed the class, you'd be wrong nearly half the time.

---

### 3. Information Gain (The "Success" Score)

Information Gain is the **difference** in entropy before and after you make a split. It tells you: *"How much did I clean up the mess?"*

* **The Formula:** $Information\ Gain = Entropy_{before} - Entropy_{after}$
* **Your Value (0.0):** This is a "Fail." It means that after you split the data using a specific feature, the two new buckets were **just as messy** as the original one.
* *Example:* If you try to separate "Boys" from "Girls" based on "Number of legs," everyone has 2 legs. The split doesn't help you distinguish them at all. Information Gain = 0.



---

### Summary Table: Comparing the Three

| Metric | Ideal Value (Pure) | Worst Value (Mixed) | Simple Definition |
| --- | --- | --- | --- |
| **Entropy** | 0.0 | 1.0 | How much "information" or "surprise" is in the data? |
| **Gini Impurity** | 0.0 | 0.5 | What is the chance of a random misclassification? |
| **Information Gain** | **Higher is Better** | 0.0 | How much did this split help me separate the classes? |

---

### Why does this matter for you?

When you build a Decision Tree, the computer calculates the **Information Gain** for every single column in your data. It will choose the column with the **highest** Information Gain to be the "Root" (the first split) of your tree.

**Would you like me to show you a small manual calculation (like a 4-row dataset) so you can see exactly where these numbers come from?**

---
# Decision Tree Example in Python

Do work on decesion tree regresor and view graph.

In [50]:
# import libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.tree import DecisionTreeClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import confusion_matrix, classification_report
from sklearn.preprocessing import LabelEncoder
from sklearn.impute import SimpleImputer
import math
from sklearn.tree import DecisionTreeRegressor
from sklearn.metrics import mean_squared_error, r2_score

In [51]:
df = pd.read_csv('/content/California_Housing_Data.csv')

In [52]:
df.head()

,longitude,latitude,housing_median_age,total_rooms,total_bedrooms,population,households,median_income,median_house_value,ocean_proximity
0,-122.23,37.88,41.0,880.0,129.0,322.0,126.0,8.3252,452600.0,NEAR BAY
1,-122.22,37.86,21.0,7099.0,1106.0,2401.0,1138.0,8.3014,358500.0,NEAR BAY
2,-122.24,37.85,52.0,1467.0,190.0,496.0,177.0,7.2574,352100.0,NEAR BAY
3,-122.25,37.85,52.0,1274.0,235.0,558.0,219.0,5.6431,341300.0,NEAR BAY
4,-122.25,37.85,52.0,1627.0,280.0,565.0,259.0,3.8462,342200.0,NEAR BAY


In [53]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 20640 entries, 0 to 20639
Data columns (total 10 columns):
 #   Column              Non-Null Count  Dtype  
---  ------              --------------  -----  
 0   longitude           20640 non-null  float64
 1   latitude            20640 non-null  float64
 2   housing_median_age  20640 non-null  float64
 3   total_rooms         20640 non-null  float64
 4   total_bedrooms      20433 non-null  float64
 5   population          20640 non-null  float64
 6   households          20640 non-null  float64
 7   median_income       20640 non-null  float64
 8   median_house_value  20640 non-null  float64
 9   ocean_proximity     20640 non-null  object 
dtypes: float64(9), object(1)
memory usage: 1.6+ MB


In [54]:
df.sample(10)

,longitude,latitude,housing_median_age,total_rooms,total_bedrooms,population,households,median_income,median_house_value,ocean_proximity
3457,-118.44,34.31,14.0,4151.0,941.0,3163.0,915.0,4.0301,154300.0,<1H OCEAN
19526,-120.97,37.65,16.0,3960.0,716.0,1776.0,724.0,3.9886,137500.0,INLAND
20552,-121.80,38.68,11.0,3851.0,892.0,1847.0,747.0,3.4331,120600.0,INLAND
5403,-118.43,34.03,36.0,1552.0,388.0,867.0,352.0,3.6467,346700.0,<1H OCEAN
13149,-121.37,36.89,21.0,2471.0,473.0,1753.0,451.0,4.0250,293800.0,INLAND
7011,-118.07,33.97,32.0,3400.0,826.0,3017.0,793.0,2.4607,155600.0,<1H OCEAN
3237,-119.56,36.10,29.0,424.0,78.0,284.0,73.0,1.5313,43800.0,INLAND
3955,-118.62,34.20,32.0,3233.0,553.0,1678.0,545.0,5.0025,234900.0,<1H OCEAN
7230,-118.14,34.02,44.0,1715.0,460.0,1740.0,423.0,2.7019,153300.0,<1H OCEAN
14417,-117.24,32.79,25.0,2135.0,691.0,566.0,320.0,2.6902,212500.0,NEAR OCEAN


In [55]:
df.isnull().sum()


,0
longitude,0
latitude,0
housing_median_age,0
total_rooms,0
total_bedrooms,207
population,0
households,0
median_income,0
median_house_value,0
ocean_proximity,0


In [56]:
df['total_bedrooms'] = df['total_bedrooms'].fillna(df['total_bedrooms'].median())


In [57]:
df.isnull().sum()

,0
longitude,0
latitude,0
housing_median_age,0
total_rooms,0
total_bedrooms,0
population,0
households,0
median_income,0
median_house_value,0
ocean_proximity,0


In [58]:
df = pd.get_dummies(df, columns=['ocean_proximity'], drop_first=True)


In [59]:
X = df.drop("median_house_value", axis=1)
y = df["median_house_value"]


In [60]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)


In [61]:
model = DecisionTreeRegressor(random_state=42)
model.fit(X_train, y_train)


DecisionTreeRegressor(random_state=42)

In [62]:
y_pred = model.predict(X_test)


In [63]:
mse = mean_squared_error(y_test, y_pred)
r2 = r2_score(y_test, y_pred)

print("Mean Squared Error:", mse)
print("R² Score:", r2)


Mean Squared Error: 4865868836.95906
R² Score: 0.6286755571919975


In [64]:
from sklearn.model_selection import GridSearchCV

param_grid = {
    'max_depth': [5, 10, 15, 20, 30],
    'min_samples_leaf': [1, 5, 10, 20]
}

grid_search = GridSearchCV(
    DecisionTreeRegressor(random_state=42),
    param_grid,
    cv=5,
    scoring='r2',
    n_jobs=-1
)

grid_search.fit(X_train, y_train)

best_model = grid_search.best_estimator_
print("Best Params:", grid_search.best_params_)

# Evaluate
y_pred = best_model.predict(X_test)
print("New MSE:", mean_squared_error(y_test, y_pred))
print("New R² Score:", r2_score(y_test, y_pred))


Best Params: {'max_depth': 15, 'min_samples_leaf': 20}
New MSE: 3397567434.1529202
New R² Score: 0.7407246523361102


In [65]:
from sklearn.ensemble import RandomForestRegressor

rf = RandomForestRegressor(n_estimators=100, max_depth=15, random_state=42)
rf.fit(X_train, y_train)
y_pred_rf = rf.predict(X_test)

print("RF MSE:", mean_squared_error(y_test, y_pred_rf))
print("RF R²:", r2_score(y_test, y_pred_rf))


RF MSE: 2451499904.1651745
RF R²: 0.8129210082598731


In [66]:
from xgboost import XGBRegressor

xgb = XGBRegressor(n_estimators=100, learning_rate=0.1, max_depth=5, random_state=42)
xgb.fit(X_train, y_train)
y_pred_xgb = xgb.predict(X_test)

print("XGB MSE:", mean_squared_error(y_test, y_pred_xgb))
print("XGB R²:", r2_score(y_test, y_pred_xgb))


XGB MSE: 2551208091.55228
XGB R²: 0.8053120717337395


In [ ]:
# save the decision tree classifier
from sklearn.tree import export_graphviz
export_graphviz(model, out_file='./saved_models/Decision_tree_03.dot', feature_names=X.columns, filled=True, rounded=True)